In [43]:
from datasets import load_dataset
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")

Resolving data files: 100%|██████████| 18/18 [00:00<00:00, 36631.48it/s]


In [44]:
filtered = dataset.filter(lambda x: x["task"] == "splice_sites_all")

In [37]:
print(filtered["train"])

Dataset({
    features: ['sequence', 'name', 'label', 'task'],
    num_rows: 30000
})


In [38]:
print(set(filtered["train"]["label"]))
seq = filtered["train"]["sequence"][0]
print(seq)

{0, 1, 2}
GTACATGCCCTAGGCCCTGGTATATGGTGGTGGCTACTGAAAATTCACTTGTAAGATTTGTACACCCATATTCACAGCAGTATTATTCACAATAGCCAAAGGGTGGAAGCAACCCAGGTATCCACCGAAGGATGAAAGGATATATGAAATGCAGCACATACTCACAATGGAATTCAGCTTTCAAAAGGAAGGAAGCTCTGACACACGCCACCACACGGATGAAGCTTGAGAACACTGTGCCATGTGAAATAAGGCAGTCACAACAGGCCAAATACTTCATGACTCCACTTACATGAGCTACCTAAAGTAGTCAAATTCATAGAAATAGAAACTAAAGTAGGGCAGGGGAGAGACTGAAAAAAGACGGGTACAGGTTGTTTTGGGAAAAAACCTTTGGAGATATATGGTGGTGATCGTTGCACAACAATGTGAATGTTCTTAATTTCACTAAACTGTACACTTAAAAATGGTTAAAATGGCTAATTTTAATTATGTATATTTTACTACAATTAAAAAACAAAACTTAAGTATATTGAGGCCGGGCGCGATGGCTCATGCATGTAATCTCAGCACTTTGGGAGGCCGAGGCGGGTGGATCAC


In [46]:
# Busca GT/AG en una ventana alrededor del centro

import collections
from datasets import load_dataset

train = filtered["train"]

WINDOW = 3  # ± nucleótidos desde el centro

dinuc_counts = {0: collections.Counter(),
                1: collections.Counter(),
                2: collections.Counter()}

for seq, label in zip(train["sequence"], train["label"]):
    mid = len(seq) // 2
    region = seq[mid - WINDOW : mid + WINDOW]
    # todos los dinucleótidos de la ventana
    for i in range(len(region) - 1):
        dinuc_counts[label][region[i:i+2]] += 1

for label, counts in dinuc_counts.items():
    total = sum(counts.values())
    gt = counts["GT"]
    ag = counts["AG"]
    top5 = counts.most_common(5)
    print(f"\nLabel {label}  (total dinucs={total:,})")
    print(f"  GT: {gt:>6,} ({100*gt/total:.1f}%)")
    print(f"  AG: {ag:>6,} ({100*ag/total:.1f}%)")
    print(f"  top-5: {top5}")


Label 0  (total dinucs=45,000)
  GT:  2,961 (6.6%)
  AG:  9,964 (22.1%)
  top-5: [('AG', 9964), ('CA', 6716), ('GG', 5418), ('GA', 4033), ('TA', 3263)]

Label 1  (total dinucs=45,000)
  GT:  9,104 (20.2%)
  AG:  6,244 (13.9%)
  top-5: [('GT', 9104), ('GG', 8192), ('AG', 6244), ('TA', 5647), ('TG', 5049)]

Label 2  (total dinucs=45,000)
  GT:  2,297 (5.1%)
  AG:  3,317 (7.4%)
  top-5: [('TT', 3745), ('AA', 3658), ('TG', 3392), ('AG', 3317), ('CT', 3284)]
